# A setup for AI-assisted programming

This notebook describes how you can setup a system to use agents powered by large language models (LLMs) to assist you in programming tasks.

## Configuring access to the Imperial HX3 cluster

During these workshops we will be using LLMs which will be running on College infrastructure, namely the [HX3 cluster](https://icl-rcs-user-guide.readthedocs.io/en/latest/hpc/hx3/).
There are many reasons as to why one would choose to run one's own LLM, two important ones are:

* Data safety: your data is not shared with 3<sup>rd</sup> parties. This is very important in a research context, as you want to share your findings via a publication not via an LLM which will not attribute it to you.
* No need to pay a subscription to companies potentially engaged in unethical practices.

You should already be registered on the HX3 system. You now need to create a token (also called an API key) which will allow your agent to connect to the LLM running on HX3. There are 2 steps to do this, first generate a Client_ID/Client_secret pair, and then from those generate the actual token.

### Generate Client_ID/Client_secret (Access Key)

1. First connect to HX3 Run:ai platform: [https://runai.hx3.hpc.ic.ac.uk](https://runai.hx3.hpc.ic.ac.uk/).
2. You can ignore all the information on the screen and click the avatar on the top right corner of the screen, and choose the option *Settings*.
3. Choose the option *Access Keys*, and press the link `+ Access Key`.
4. You can name this access key whatever you like. For example: ai-programming-workshop
5. On the next screen you will see the Client_ID/Client_secret pair that was generated (you will need to click on the 'eye' icon to see the client secret. Copy the Client_ID and Client_secret values to the code cell below. After you copy the value you can press `Done` and close the Run:ai webpage.

In [ ]:
runai_client_id=""
runai_client_secret=""

### Generating tokens/API key

The code cell below uses your Client_ID/Client_secret pair defined above to connect to HX3 and generate a token.

In [ ]:
import urllib.request
import urllib.parse
import json

data = {
    "grantType": "client_credentials",
    "clientId": runai_client_id,
    "clientSecret": runai_client_secret
}

req = urllib.request.Request(
    "https://runai.hx3.hpc.ic.ac.uk/api/v1/token",
    data=json.dumps(data).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(req) as response:
    response_data = json.loads(response.read().decode("utf-8"))
    
token = response_data["accessToken"]
print(token)

Tokens generated by HX3 expire after about one hour. When this happens you may need to run the code above to generate another token.

Note that your Client_ID/Client_secret pair is sensitive information, equivalent to login/password pair, so you want to keep this information and notebook private and safe.

## Installing software

For the purpose of these workshops we will install 2 pieces of software:

**Visual Studio Code (aka VScode)**
: This is a popular integrated development environment (IDE) developed by Microsoft. One could use one of many other existing IDEs.

**Continue VS Code extension**
: This is our or agent, which passes on our queries to the LLM and relay its output. Importantly, the agent does not simply display the text output by the LLM, but it is also able to perform actions, such as creating new files, or be asked to read the content of files or websites.

### Installing Visual Studio Code

[Download](https://code.visualstudio.com/download) and run the installer for your platform.

When asked about the installation of the Copilot agent and use of a Github account it is safe to say no. We will be using a different agent.

### Installing the Continue VS Code extension

To install extensions in VScode, go to *View* on the top menu and select *Extensions*. Search for "Continue - open source AI code agent", check that the publisher is "Continue", and install the extension.

A new icon will appear on the left-hand-side bar (1) and you can use it to invoke the Continue extension. The configuration of continue is primarily done by editing the configuration file. To access it:

2) on the Continue window top right corner click on the cogwheel to configure the extension;
3) then on the left-hand-side panel choose "Configs";
4) then the Main Config cogwheel and you will be shown an almost empty *config.yaml* file.

![VS Code screenshot with the configuration of Continue](resources/VScode_screenshot.png "Configuration of Continue")

Copy the following content to the *config.yaml* file such that it looks like the following:

    name: Main Config
    version: 1.0.0
    schema: v1
    
    models:
      - name: HX3 devstral
        provider: openai
        model: mistralai/Devstral-Small-2-24B-Instruct-2512
        apiBase: https://devstral-runai-chemistry-tests.runai-inference.hx3.hpc.ic.ac.uk/v1
        apiKey:
        roles: [chat, edit, apply, autocomplete]
        capabilities: ["tool_use"]
      
      - name: HX3 qwen
        provider: openai
        model: Qwen/Qwen3-0.6B
        apiBase: https://qwen-runai-chemistry-tests.runai-inference.hx3.hpc.ic.ac.uk/v1
        apiKey:
        roles: [chat, edit, apply, autocomplete]
        capabilities: ["tool_use"]

You should copy the token generated above in this notebook to the configuration file as the *apiKey*.

In this configuration we are setting up 2 LLMs, one from the Devstral family of models (especially trained for programming tasks) and a small model from the Qwen3 family (a generalist model). Although the *provider* option is set to openai, we are not using openai models or openai's infrastructure, both models are being run on the HX3 cluster. We are however using a openai compatible protocol to interface with the models.

As mentioned above, the token will expire after one hour, in which case you need to generate another one and copy it to the *config.yaml* file.

There are many aspects of the agent behaviour and workflow that can be configured in the *config.yaml*. You can find a reference for all options available on the [Continue website](https://docs.continue.dev/reference).